
# Урок 2. Датасеты и Feature Engineering  
Мини-курс **«ML для аналитиков»** — практический ноутбук.

**Что делаем:**  
1) создаём мини-датасет прямо в ноутбуке;  
2) смотрим на качество и структуру данных;  
3) выполняем простую очистку и подготовку признаков;  
4) делим данные на train/test и обсуждаем зачем это нужно;  
5) строим пару быстрых графиков для обсуждения.
-----
> Цель занятия — увидеть, **как аналитик готовит данные для модели**.


In [ ]:
# 1) Импорты
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
print("Готово: pandas, numpy, train_test_split, matplotlib")


## 1. Создаём мини-датасет (табличка)
- Каждая строка — клиент.
- Столбцы — признаки (features).
- `churn` — целевая переменная (что хотим предсказать).


In [ ]:
# 2) Создаём мини-датасет прямо здесь (без CSV)
data = {
    "client_id": list(range(1, 16)),
    "age": [22, 35, 58, 41, 37, np.nan, 29, 46, 51, 34, 40, 67, 31, 25, 43],
    "gender": ["F","M","M","F","M","F","M","M","F","F","M","F","M","M","F"],
    "num_purchases": [5, 12, 2, 7, 4, 9, 1, 15, 11, 8, 3, 6, 0, 10, 13],
    "avg_bill": [900, 2500, 450, 1300, 850, 2100, 300, 2700, 1900, 1600, 700, 1200, 0, 1800, 2400],
    "support_tickets": [0, 1, 0, 1, 2, 0, 0, 3, 1, 1, 0, 2, 0, 0, 1],
    "churn": [0,0,1,0,0,1,1,0,0,0,1,0,1,0,0]  # 1 = ушёл
}
df = pd.DataFrame(data)
print("Размеры:", df.shape)
df.head()


## 2. Быстрый обзор качества данных
Что всегда полезно посмотреть перед любыми моделями:
- типы столбцов (`info()`);
- базовые описательные статистики (`describe()`);
- пропуски и странные значения.


In [ ]:
print("Информация о данных:")
display(df.info())

print("\nОписательная статистика:")
display(df.describe(include="all"))

print("\nПропуски по столбцам:")
display(df.isna().sum())


## 3. Очистка и подготовка
- Заполним пропуски по `age` медианой (как простой и быстрый подход).
- Уберём записи с `avg_bill = 0` как заведомо некорректные для этого примера.
- Преобразуем `gender` в числовой формат.
> Это **демо-подходы**. В реальном проекте решения принимаются по контексту бизнеса.


In [ ]:
# Заполним пропуск в возрасте медианой
median_age = df["age"].median()
df["age"] = df["age"].fillna(median_age)

# Удалим строки с нулевым avg_bill
df = df[df["avg_bill"] > 0].copy()

# Категориальный признак gender -> числовой
df["gender_num"] = df["gender"].map({"F": 0, "M": 1})

print("После очистки размеры:", df.shape)
df.head()


## 4. Train/Test Split (деление на обучение/проверку)
Это как **встроенный A/B-тест для модели**: учимся на одной части данных и проверяем на другой,
чтобы оценить, как модель будет вести себя на будущем, невиданном ранее.


In [ ]:
features = ["age", "gender_num", "num_purchases", "avg_bill", "support_tickets"]
target = "churn"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Размеры: X_train={X_train.shape}, X_test={X_test.shape}")


## 5. Быстрые визуализации для обсуждения
> Важно: графики здесь — для разговора «что может быть полезным признаком», а не для красоты.


In [ ]:
# Гистограмма среднего чека
plt.figure(figsize=(6,4))
plt.hist(df["avg_bill"], bins=8)
plt.title("Распределение среднего чека (avg_bill)")
plt.xlabel("avg_bill")
plt.ylabel("count")
plt.show()

In [ ]:
# Точечный график: num_purchases vs. avg_bill
plt.figure(figsize=(6,4))
plt.scatter(df["num_purchases"], df["avg_bill"])
plt.title("Кол-во покупок vs. средний чек")
plt.xlabel("num_purchases")
plt.ylabel("avg_bill")
plt.show()


## 6. Для обсуждения / мини-задания
1) Какие **новые признаки** вы бы добавили (как аналитики)? Примеры:  
   - «среднее кол-во покупок за последние N дней»,  
   - «доля новых категорий покупок»,  
   - «время с последней активности».  
2) Что делать с выбросами и редкими категорями?  
3) Чем заполнение медианой лучше/хуже удаления строк?  
4) Какие столбцы в **наших данных** могут стать хорошими фичами для прогноза оттока?

---

📌 На следующем занятии: запустим **первую простую модель** и посмотрим на метрики качества.
